# 01 Homework 04 ETM — IFC Semantic Relationships

**Task 1 of the Final Assignment**

This notebook loads the Brownstone IFC model (`Brownstone.ifc`, project “6 Great Circle”) and
builds a **semantic relationships graph** from it using TopologicPy.
Every IFC element becomes a node; every IFC relationship (‘Rel’ entity) becomes an edge.

**Building summary**
- Schema: IFC4
- Storeys: Ground Floor (Z≈0), Level 1 (Z=2.08 m), Level 2 (Z=5.71 m), Level 3 (Z=8.48 m), Roof (Z=11.38 m)
- 40 walls · 5 floor slabs · 33 wall openings · 241 building element proxies

**Key IFC relationships present**
| Relationship | Meaning |
|---|---|
| `IfcRelContainedInSpatialStructure` | Element → storey containment |
| `IfcRelVoidsElement` | Wall → opening cut |
| `IfcRelDefinesByType` | Element → element-type link |
| `IfcRelAggregates` | Spatial decomposition (site → building → storey) |

## 1. Import the needed TopologicPy classes

In [1]:
from topologicpy.Cluster import Cluster
from topologicpy.Color import Color
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Color mappings

Two mappings control the visual appearance of the semantic graph:

- **`ifc_color_mapping`** — colors graph *nodes* by IFC element type (e.g. walls red, slabs green)
- **`rels_color_mapping`** — colors graph *edges* by IFC relationship type

The mappings below are tuned to the entity types actually present in the Brownstone IFC.

In [4]:
# ── Graphic Style (shared look with 02_Homework04_Graphs) ─────────────────────
GRAPH_NODE_SIZE_KEY  = "size"
GRAPH_NODE_COLOR_KEY = "color"
GRAPH_NODE_LABEL_KEY = "label"
GRAPH_EDGE_COLOR     = "#6F6F6F"
GRAPH_EDGE_WIDTH     = 3
GRAPH_BG_DARK        = "white"
GRAPH_BG_LIGHT       = "white"

# --- Semantic relationship edge colours ---
semantic_rels_color_mapping = {
    "IfcRelContainedInSpatialStructure": "#31688E",  # storey containment  (blue)
    "IfcRelVoidsElement":                "#E64B5D",  # wall opening cuts   (red)
    "IfcRelAggregates":                  "#FDE725",  # spatial hierarchy   (yellow)
}

# --- Spatial relationship edge colours (Part B) ---
spatial_rels_color_mapping = {
    "contains":     "#FF0000",
    "coveredBy":    "#0000C8",
    "covers":       "#0000C8",
    "crosses":      "#0098FF",
    "disjoint":     "#2CFF96",
    "equals":       "#97FF00",
    "overlaps":     "#FFEA00",
    "touches":      "#550E55",
    "within":       "#FF0000",
    "near":         "#AAAAAA",
    "intermediate": "#666666",
    "far":          "#000000",
}

# --- IFC element node colours (shared by both parts) ---
# NOTE: background is now white (shared look), so storeys are a visible grey
# instead of the old white-on-black, and the fallbacks below use GRAPH_EDGE_COLOR.
ifc_color_mapping = {
    "ifcsite":                    "#AAAAAA",
    "ifcbuilding":                "#888888",
    "ifcbuildingstorey":          "#555555",
    "ifcwall":                    "#F8765C",
    "ifcwallstandardcase":        "#E64B5D",
    "ifcslab":                    "#B4DE2C",
    "ifcopeningelement":          "#35B779",
    "ifcbuildingelementproxy":    "#26828E",  # furniture / misc elements
}

### Import Config

Loads shared settings from `config.py` (the `REPO_ROOT` path and the camera presets),
so this notebook has no machine-specific absolute paths. The loader finds `config.py`
regardless of the kernel's working directory.

In [5]:
import os, sys

# Locate the folder that holds config.py, regardless of the kernel's working
# directory. (VSCode often starts the kernel at the workspace root rather than
# this notebook's folder, so `os.getcwd()` is not reliably the Notebooks dir.
# Note: `__file__` does not exist in a Jupyter kernel.)
def _find_config_dir(start):
    d = os.path.abspath(start)
    while True:
        for cand in (d, os.path.join(d, "Homework04", "Notebooks")):
            if os.path.isfile(os.path.join(cand, "config.py")):
                return cand
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(f"config.py not found searching up from {os.path.abspath(start)}")
        d = parent

sys.path.insert(0, _find_config_dir(os.getcwd()))
from config import *

## 5. Specify the IFC file path

In [6]:
ifc_file_path = os.path.join(REPO_ROOT, "Homework04", "Brownstone.ifc")

## 6. Import the IFC file as a semantic graph

`Graph.ByIFCPath` parses the IFC file and builds a graph where every IFC product is a node and every IFC relationship is an edge.

**Why `importMode="topology"`?**  
The Brownstone export contains 241 `IfcBuildingElementProxy` elements (furniture / misc), which would make full geometry import very slow. Topology mode places nodes from IFC entity positions directly — much faster and still accurate for semantic analysis.

**Why `includeTypes` and `includeRels`?**  
Without filtering, the graph also loads `IfcRelDefinesByProperties` (985 instances) and `IfcRelDefinesByType` (245 instances), which create a star pattern where every element fans out to ONE shared type/property-set node — the "exterior node" that appears disconnected from the rest of the graph. Filtering to only the meaningful architectural relationships fixes this.

In [7]:
graph = Graph.ByIFCPath(
    ifc_file_path,
    importMode="topology",
    dictionaryMode="basic",
    storeBREP=True,
    includeTypes=[
        "IfcWall", "IfcSlab", "IfcOpeningElement",
        "IfcBuildingElementProxy",
        "IfcBuildingStorey", "IfcBuilding", "IfcSite",
    ],
    includeRels=[
        "IfcRelContainedInSpatialStructure",
        "IfcRelVoidsElement",
        "IfcRelAggregates",
    ],
)

print(f"Graph loaded: {len(Graph.Vertices(graph))} nodes, {len(Graph.Edges(graph))} edges")

Graph loaded: 327 nodes, 326 edges


## 7. Extract bounding boxes and apply colors

For each graph node:
- Retrieve the stored BREP string, reconstruct the geometry, compute its axis-aligned bounding box
- Look up the node’s `IFC_type` in `ifc_color_mapping` and store `color` + `size` in the node dictionary

For each graph edge:
- Look up the edge’s `IFC_type` in `rels_color_mapping` and store `color` + `width`

In [8]:
vertices = Graph.Vertices(graph)
boxes = []

for v in vertices:
    d = Topology.Dictionary(v)
    brep = Dictionary.ValueAtKey(d, "BREP")
    if brep:
        topology = Topology.ByBREPString(brep)
        box = Topology.BoundingBox(topology)
        boxes.append(box)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "unknown")
    color = ifc_color_mapping.get(ifc_type.lower(), "#AAAAAA")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, color])
    v = Topology.SetDictionary(v, d)

edges = Graph.Edges(graph)
for e in edges:
    d = Topology.Dictionary(e)
    ifc_rel = Dictionary.ValueAtKey(d, "IFC_type")
    color = semantic_rels_color_mapping.get(ifc_rel, GRAPH_EDGE_COLOR)
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [GRAPH_EDGE_WIDTH, color])
    e = Topology.SetDictionary(e, d)

print(f"Bounding boxes extracted: {len(boxes)}")

Bounding boxes extracted: 0


## 8. Show the semantic graph with bounding box geometry

The 3D view shows element bounding boxes (semi-transparent) overlaid with the semantic graph.
Node colors identify element types; edge colors identify relationship types.

In [9]:
Topology.Show(
    boxes, graph,
    faceOpacity=0.1,
    sagitta=0.15,
    absolute=False,
    backgroundColor=GRAPH_BG_LIGHT,
    vertexSizeKey=GRAPH_NODE_SIZE_KEY,
    vertexColorKey=GRAPH_NODE_COLOR_KEY,
    edgeColor=GRAPH_EDGE_COLOR,
    edgeWidthKey="width",
    edgeColorKey="color",
    camera=CAM_STACK,
    center=CAM_STACK_CTR,
    up=CAM_STACK_UP,
    projection=CAM_STACK_PRJ,
    width=1000,
    height=1000,
    renderer=renderer
)

## 9. Interactive graph with Pyvis

Pyvis renders the semantic graph as an interactive 2D network in a browser tab.
Nodes are draggable; hovering shows the element type. This view makes it easy to
explore which elements cluster together and which relationships dominate the graph.

The HTML file is saved next to this notebook.

In [10]:
pyvis_out = os.path.join(REPO_ROOT, "Homework04", "Notebooks", "01_Homework04_etm_graph.html")

pyvis_graph = Graph.PyvisGraph(
    graph,
    path=pyvis_out,
    vertexSizeKey="size",
    vertexColorKey="color",
    vertexLabelKey="IFC_type",
    edgeWeightKey="width",
    edgeColorKey="color",
)
print(f"Pyvis graph saved to: {pyvis_out}")

C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Notebooks\01_Homework04_etm_graph.html
Pyvis graph saved to: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Notebooks\01_Homework04_etm_graph.html


---
# Part B: Spatial Relationships

Geometric (bounding-box) relationships between building elements — `touches`, `overlaps` — computed directly from element geometry rather than IFC relationship entities.
This reveals physical adjacency that the IFC semantic graph does not explicitly encode.

## 10. Specify element and relationship types for spatial analysis

In [11]:
# Focus on structural / envelope elements — exclude furniture proxies to keep the graph readable
includeTypes_spatial = ["IfcWall", "IfcSlab", "IfcOpeningElement"]

includeRels_spatial = ["overlaps", "touches"]

## 11. Import IFC objects and create bounding boxes

In [12]:
ifc_objects = Topology.ByIFCPath(
    ifc_file_path,
    includeTypes=includeTypes_spatial,
    dictionaryMode="basic",
)
spatial_boxes = [Topology.BoundingBox(obj) for obj in ifc_objects]
print(f"Imported {len(ifc_objects)} IFC objects → {len(spatial_boxes)} bounding boxes")

IFCFastTopology.Parse - Parsed 14561 entities in 0.504s.
IFCFastTopology.TopologiesByEntities - Created 61 topologies; skipped 17 products in 1.702s.
Imported 61 IFC objects → 61 bounding boxes


## 12. Show the simplified IFC model (bounding boxes)

In [13]:
Topology.Show(
    spatial_boxes,
    faceOpacity=0.2,
    backgroundColor=GRAPH_BG_LIGHT,
    camera=CAM_STACK,
    center=CAM_STACK_CTR,
    up=CAM_STACK_UP,
    projection=CAM_STACK_PRJ,
    width=1000, height=1000,
    renderer=renderer
)

## 13. Create the spatial relationships graph

In [14]:
spatial_graph = Graph.BySpatialRelationships(spatial_boxes, include=includeRels_spatial)
print(f"Spatial graph: {len(Graph.Vertices(spatial_graph))} vertices, {len(Graph.Edges(spatial_graph))} edges")

Spatial graph: 61 vertices, 161 edges


## 14. Apply colors to spatial graph nodes and edges

In [15]:
spatial_edges = Graph.Edges(spatial_graph)
for e in spatial_edges:
    d = Topology.Dictionary(e)
    rel_fwd = Dictionary.ValueAtKey(d, "relFwd")
    color = spatial_rels_color_mapping.get(rel_fwd, GRAPH_EDGE_COLOR)
    d = Dictionary.SetValuesAtKeys(d, ["color", "width"], [color, GRAPH_EDGE_WIDTH])
    e = Topology.SetDictionary(e, d)

spatial_vertices = Graph.Vertices(spatial_graph)
for v in spatial_vertices:
    d = Topology.Dictionary(v)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "unknown")
    color = ifc_color_mapping.get(ifc_type.lower(), "#AAAAAA")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, color])
    v = Topology.SetDictionary(v, d)

## 15. Show the spatial relationships graph

In [16]:
Topology.Show(
    spatial_boxes, spatial_graph,
    faceOpacity=0.1,
    sagitta=0.15,
    absolute=False,
    backgroundColor=GRAPH_BG_LIGHT,
    vertexSizeKey=GRAPH_NODE_SIZE_KEY,
    vertexColorKey=GRAPH_NODE_COLOR_KEY,
    edgeColor=GRAPH_EDGE_COLOR,
    edgeWidthKey="width",
    edgeColorKey="color",
    camera=CAM_STACK,
    center=CAM_STACK_CTR,
    up=CAM_STACK_UP,
    projection=CAM_STACK_PRJ,
    width=1000, height=1000,
    renderer=renderer
)

## 16. Pyvis interactive view of the spatial graph

In [17]:
pyvis_spatial_out = os.path.join(REPO_ROOT, "Homework04", "Notebooks", "01_Homework04_etm_spatial_graph.html")

pyvis_spatial = Graph.PyvisGraph(
    spatial_graph,
    path=pyvis_spatial_out,
    vertexSizeKey="size",
    vertexColorKey="color",
    vertexLabelKey="IFC_type",
    edgeWeightKey="width",
    edgeColorKey="color",
)
print(f"Spatial Pyvis graph saved to: {pyvis_spatial_out}")

C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Notebooks\01_Homework04_etm_spatial_graph.html
Spatial Pyvis graph saved to: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Notebooks\01_Homework04_etm_spatial_graph.html
